# 08 — Modules, Packages, and Imports

Goal: understand how Python finds code (`sys.path`), how packages work, and how to avoid import traps.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Module vs package

- A **module** is a `.py` file.
- A **package** is a directory of modules (usually with `__init__.py`).

Imports are cached in `sys.modules`.

In [ ]:

import sys
print("first 5 sys.path entries:")
for p in sys.path[:5]:
    print(" ", p)


## 2.
L2: Absolute vs relative imports

In packages, prefer absolute imports from your top-level package:
`from my_project.util import foo`

Relative imports (`from .util import foo`) can be correct inside packages, but are easier to break in notebooks/scripts.

## 3.
L3: `__name__`, `__main__`, and executable packages

- A file run directly has `__name__ == "__main__"`.
- `python -m pkg` executes `pkg/__main__.py`.

## 4.
L4: Hands-on: create a tiny package and import it

We’ll create this structure on disk:

```
tmp_pkg/
  demo_pkg/
    __init__.py
    mathy.py
    __main__.py
```

In [ ]:

import tempfile
from pathlib import Path
import importlib
import sys

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name) / "tmp_pkg"
pkg = root / "demo_pkg"
pkg.mkdir(parents=True)

(pkg / "__init__.py").write_text('VERSION = "0.1.0"\n', encoding="utf-8")
(pkg / "mathy.py").write_text('def add(a,b): return a+b\n', encoding="utf-8")
(pkg / "__main__.py").write_text('from .mathy import add\nprint(add(2,3))\n', encoding="utf-8")

sys.path.insert(0, str(root))  # make tmp_pkg importable

demo_pkg = importlib.import_module("demo_pkg")
mathy = importlib.import_module("demo_pkg.mathy")
print("VERSION:", demo_pkg.VERSION)
print("add:", mathy.add(10, 20))


## 5.
L5: Import cycles (why they happen)

Import cycles occur when:
- module A imports B at import time
- module B imports A at import time

Fixes:
- refactor shared code into a third module
- move imports inside functions (only when it truly breaks a cycle)
- reduce module-level side effects

## 5.
L5: Reloading modules (for interactive work)

In notebooks, you may want to reload during development:
`importlib.reload(module)`

In production, avoid reload; use tests + restart process.

In [ ]:

import importlib
mathy = importlib.reload(mathy)
print(mathy.add(1, 1))


## 6.
L6: `__all__` and star imports

Avoid `from x import *` in real code.
`__all__` controls what star-import exports, but explicit imports are clearer.

## 7.
L7: Exercises

1. Add a function `mul(a,b)` to `mathy.py`, reload, and call it.
2. Print all loaded modules containing the substring `"demo_pkg"` from `sys.modules`.
3. Explain when you would use `python -m ...` instead of `python script.py`.

In [ ]:

# cleanup tempdir (also removes it from sys.path if you want)
tmp.cleanup()


## 8.
L8: `if TYPE_CHECKING` for optional imports

When type hints would otherwise create import-time cycles or heavy dependencies:

```python
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from big_module import BigType
```